# Puckworks — Guided Pull Laboratory (browser, no terminal)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trbrewer/puckworks/blob/main/notebooks/guided_pull_laboratory_colab.ipynb)

Run a bounded espresso **Laboratory** shot in your browser — no install, no terminal, no code to type.
Fill in the short **form**, then press the single **▶ Run the Laboratory** button.

This is an honest research tool, **not** a digital twin, a recipe optimizer, or a taste predictor.
Component self-checks are each a model's *own* reference case — not a prediction of your shot.
It runs **privately** in your Colab runtime; that does not make any model cleared for public hosting.

**Development preview (0.4.0.dev0)** — not the v0.3.0 public release.

## 1. Set up (runs automatically — nothing to type)

In [ ]:
# This runs automatically — you do NOT type anything here.
# DEVELOPMENT PREVIEW: the Guided Pull Laboratory is 0.4.0.dev0 (NOT the v0.3.0 public release).
# It installs an EXACT, immutable pinned commit of the source (no mutable "latest main").
import os, subprocess, sys, hashlib

PIN_COMMIT = "59350f98d0ef8df1707007dee3d0fb57e3a4c08b"
INSTALL_SOURCE = f"git+https://github.com/trbrewer/puckworks@{PIN_COMMIT}"

override = os.environ.get("PUCKWORKS_WHEEL", "").strip()
if override:
    # Hermetic / CI mode: install a locally built candidate wheel, no network.
    if not os.path.isfile(override):
        raise SystemExit("PUCKWORKS_WHEEL is set but is not a file: " + override)
    h = hashlib.sha256(open(override, "rb").read()).hexdigest()
    print("Hermetic mode — installing local candidate wheel")
    print("  wheel sha256:", h)
    INSTALL_SOURCE = override
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", override], check=True)
else:
    # DEVELOPMENT PREVIEW — pinned to an immutable commit (the commit SHA is the integrity pin).
    print("DEVELOPMENT PREVIEW — installing pinned commit", PIN_COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", INSTALL_SOURCE], check=True)
print("Install complete. install_source =", INSTALL_SOURCE)

## 2. Environment check

In [ ]:
import puckworks
print("puckworks version :", puckworks.__version__)
print("pinned commit     :", PIN_COMMIT)
print("install source    :", INSTALL_SOURCE)
print("components         :", len(puckworks.components()))
# This is the DEVELOPMENT PREVIEW (0.4.0.dev0) — do not read it as the v0.3.0 public release.
assert puckworks.__version__ == "0.4.0.dev0", puckworks.__version__

## 3. Choose your shot

Adjust the form on the right. `pv19_named` is a fixed reference recipe. A grinder dial is **not** a universal particle size, so there is no grind→size control here.

In [ ]:
#@title Your shot { display-mode: "form", run: "auto" }
#@markdown Pick a starting recipe and adjust the three numbers. Then run the **Run the Laboratory** cell below.
preset = "pv19_named" #@param ["pv19_named", "guided_v1"]
dose_g = 20.0 #@param {type:"number"}
target_beverage_g = 40.0 #@param {type:"number"}
pressure_bar = 9.0 #@param {type:"number"}
#@markdown **What would you like to do?**
experience_mode = "Guided shot only" #@param ["Guided shot only", "Guided shot + component self-checks", "Catalog only (browse models, run nothing)"]

### Advanced (optional)

In [ ]:
#@title Advanced (optional) { display-mode: "form", run: "auto" }
#@markdown Most people can leave these as-is.
brew_temperature_c = 93.0 #@param {type:"number"}
#@markdown What should happen outside the evidence range?
outside_evidence_range = "Show the result with a warning" #@param ["Show the result with a warning", "Do not run outside the evidence range"]

## 4. Run the Laboratory

Press ▶ on the cell below. It builds one explicit, bounded request and runs it **privately** (`LOCAL_PRIVATE`).

In [ ]:
#@title ▶ Run the Laboratory { display-mode: "form" }
# Press the play button on THIS cell. It maps your choices to one explicit, bounded request and runs it
# PRIVATELY in this notebook runtime (execution context LOCAL_PRIVATE). "Private" means: it runs in the
# Colab runtime you control, and it does NOT mean the model is cleared for public hosting.
from puckworks.product import lab, lab_service, lab_explorer

_overrides = {"dose_g": float(dose_g), "target_beverage_g": float(target_beverage_g),
              "pressure_bar": float(pressure_bar), "brew_temperature_c": float(brew_temperature_c)}
_domain_policy = "strict" if outside_evidence_range.startswith("Do not run") else "warn"

# each experience mode maps to ONE explicit, finite request (never "select every future runner")
mode = experience_mode
lab_result = None
catalog = None
if mode.startswith("Catalog only"):
    # producer-free: browse the full model library, run NOTHING
    catalog = lab_explorer.explorer_catalog()
    print("Catalog only — browsing", catalog["n_components"], "models; no scientific producer ran.")
else:
    if mode.startswith("Guided shot + component"):
        req = lab.ScenarioRequest(preset, overrides=_overrides, domain_policy=_domain_policy,
                                  lens_selection_policy="primary",
                                  reference_selection_policy="interactive_fast")
    else:  # "Guided shot only"
        req = lab.ScenarioRequest(preset, overrides=_overrides, domain_policy=_domain_policy,
                                  lens_selection_policy="primary", reference_selection_policy="none")
    lab_result = lab_service.execute_lab_request(req, execution_context="LOCAL_PRIVATE")
    if lab_result.blocked:
        print("This request was blocked by the rights preflight (no model ran):")
        for b in lab_result.blockers:
            print("  -", b)
    else:
        print("Ran:", ", ".join(lab_result.selected_component_ids) or "(model lens)")
print("Done. Scroll down for results.")

## 5. What was run

In [ ]:
# What was run — the honest headline, before any chart.
if catalog is not None:
    live = catalog["public_live_component_ids"]
    print("Model library:", catalog["n_components"], "models.")
    print("Publicly-runnable today (rights-cleared):", ", ".join(live) if live else "(none yet)")
    print("Nothing was executed in Catalog-only mode.")
elif lab_result is None or lab_result.blocked:
    print("No scientific result (the request was blocked or not run).")
else:
    rep = lab_result.report
    sc = rep["scenario"]
    print("Scenario:", sc["scenario_id"], "| overrides:", sc["applied_overrides"] or "none")
    dom = rep["domain"]
    if dom["blocked"]:
        print("Evidence range: OUTSIDE — the model was not run (your strict choice).")
    elif dom["warning_count"]:
        print("Evidence range: a WARNING was raised (result shown with a caveat).")
    else:
        print("Evidence range: inside the model's stated range.")
    for lens in rep.get("executed_lenses", []):
        print("\nModel that used this recipe:", lens["component_id"])
        for o in lens["observables"][:4]:
            print(f"  {o['name']}: {o['value']} {o['unit']}  [{o['role']}]")
    print("\nWhat this does NOT prove:")
    for s in rep["what_this_does_not_prove"][:4]:
        print("  -", s)
    print(rep["fidelity_ceiling"])

## 6. Plots and component self-checks

In [ ]:
# Unit-safe plots + component self-checks (each self-check is the component's OWN reference case,
# not a prediction of your shot). Skipped in Catalog-only mode.
if lab_result is not None and not lab_result.blocked:
    rep = lab_result.report
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        for panel in lab.render_data(rep):
            fig, ax = plt.subplots(figsize=(7, 3.5))
            for s in panel["series"]:
                ax.plot(panel["x"], s["y"], label=f"{s['label']} [{s['role']}]")
            ax.set_xlabel(panel["x_label"]); ax.set_ylabel(panel["y_label"])  # one unit per axis
            ax.set_title(panel["title"]); ax.legend(fontsize=7); fig.tight_layout()
            try:
                from IPython.display import display; display(fig)
            finally:
                plt.close(fig)
    except Exception as exc:
        print("(plots unavailable:", type(exc).__name__, "-", exc, ")")
    refs = rep.get("executed_reference_results", [])
    if refs:
        print("\nComponent self-checks (each is the component's own reference case):")
        for r in refs:
            print(" ", r["component_id"], "-", r["status"], "-", r.get("runtime_class"))
else:
    print("(no plots — Catalog-only or blocked)")

## 7. Public vs private

In [ ]:
# Public vs private, in plain language.
from puckworks.product import lab_explorer
live = lab_explorer.public_live_component_ids()
print("This notebook runs PRIVATELY in your Colab runtime (context: LOCAL_PRIVATE).")
print("Private inspection can run models that are not yet cleared for PUBLIC hosting.")
print("Publicly-runnable models today (affirmative rights only):", ", ".join(live) if live else "(none)")
print("grudeva2025.reduced is rights-blocked and never runs here (issue #73).")

## 8. Downloads and technical details

In [ ]:
# Download the machine-readable result (guided modes only) + technical provenance.
if lab_result is not None and not lab_result.blocked:
    from puckworks.product import lab
    rep = lab_result.report
    j = lab.artifact_json(rep); m = lab.render_markdown(rep)
    open("guided_pull_lab.json", "w").write(j)
    open("guided_pull_lab.md", "w").write(m)
    print("Wrote guided_pull_lab.json and guided_pull_lab.md")
    print("scientific_payload_sha256:", rep["integrity"]["scientific_payload_sha256"])
    print("package version:", rep["provenance"].get("package_version"))
    try:
        from google.colab import files as _f
        _f.download("guided_pull_lab.json"); _f.download("guided_pull_lab.md")
    except Exception:
        print("(not in Colab; files written to the working directory)")
else:
    print("(nothing to download — Catalog-only or blocked request)")

In [ ]:
# completion sentinel (used by CI to confirm the notebook ran end-to-end)
_ver = __import__("puckworks").__version__
if catalog is not None:
    _mode, _components, _hash = "catalog_only", ",".join(catalog["public_live_component_ids"]) or "-", "NA"
elif lab_result is not None and not lab_result.blocked:
    _mode = "guided"
    _components = ",".join(lab_result.selected_component_ids) or "lens"
    _hash = lab_result.report["integrity"]["scientific_payload_sha256"]
else:
    _mode, _components, _hash = "blocked", "-", "NA"
print(f"GUIDED_PULL_LAB_COMPLETE: {_ver} · mode={_mode} · context=LOCAL_PRIVATE · "
      f"components={_components} · sci_hash={_hash}")